# 🌿 Plant Disease Detection — Improved Pipeline
Unsupervised anomaly detection using a ResNet18 Autoencoder.

**Improvements over original:**
- Multi-scale patch extraction (32 + 64 + 128px)
- SSIM + MSE combined loss
- Adaptive thresholding per image
- Gaussian blur smoothing of anomaly maps
- Morphological closing to fill lesion holes
- Training augmentation (flip + brightness)
- LR scheduler (StepLR)
- Auto-zip of results for download

## 1. Install dependencies

In [ ]:
!pip install pytorch-msssim -q
!pip install tqdm -q

## 2. Imports

In [ ]:
import os
import cv2
import numpy as np
import random
import zipfile
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision.models as models
from pytorch_msssim import ssim

from sklearn.cluster import KMeans

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Upload & extract dataset

In [ ]:
# Upload your data_set.zip via the Colab file browser, then run:
!unzip -q data_set.zip
print("Dataset extracted.")

## 4. Dataset splitting

In [ ]:
def split_dataset(root):
    """Separate healthy from diseased image paths."""
    healthy_paths = []
    diseased_paths = []

    for folder in os.listdir(root):
        folder_path = os.path.join(root, folder)
        if not os.path.isdir(folder_path):
            continue
        if "healthy" in folder.lower():
            for img in os.listdir(folder_path):
                healthy_paths.append(os.path.join(folder_path, img))
        else:
            for img in os.listdir(folder_path):
                diseased_paths.append(os.path.join(folder_path, img))

    print(f"Healthy: {len(healthy_paths)} | Diseased: {len(diseased_paths)}")
    return healthy_paths, diseased_paths

## 5. Preprocessing

In [ ]:
def preprocess(path):
    """Load, resize to 256x256, and apply CLAHE histogram equalisation."""
    img = cv2.imread(path)
    if img is None:
        return None

    img = cv2.resize(img, (256, 256))

    # CLAHE on L channel only
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)          # IMPROVED: CLAHE instead of global equalizeHist
    lab = cv2.merge((l, a, b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    img = img / 255.0
    return img.astype("float32")

## 6. Multi-scale patch extraction (NEW)

In [ ]:
def extract_patches(img, size=64, stride=32):
    """Single-scale patch extraction."""
    patches, coords = [], []
    h, w, _ = img.shape
    for y in range(0, h - size, stride):
        for x in range(0, w - size, stride):
            patches.append(img[y:y+size, x:x+size])
            coords.append((x, y))
    return np.array(patches), coords


def extract_patches_multiscale(img, sizes=[32, 64, 128], stride_ratio=0.5):
    """
    IMPROVED: Extract patches at multiple scales, all resized to 64x64.
    Catches both fine-grained spots and large lesion regions.
    """
    all_patches, all_coords, all_sizes = [], [], []
    h, w, _ = img.shape
    for size in sizes:
        stride = max(8, int(size * stride_ratio))
        for y in range(0, h - size, stride):
            for x in range(0, w - size, stride):
                patch = img[y:y+size, x:x+size]
                patch = cv2.resize(patch, (64, 64))
                all_patches.append(patch)
                all_coords.append((x, y))
                all_sizes.append(size)
    return np.array(all_patches), all_coords, all_sizes

## 7. Model — ResNet18 Autoencoder

In [ ]:
# Encoder: frozen ResNet18 backbone (outputs 512-ch feature map)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
encoder = nn.Sequential(*list(resnet.children())[:-2])
for param in encoder.parameters():
    param.requires_grad = False


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, 2),   # 2 -> 4
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 2, 2),   # 4 -> 8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 2, 2),    # 8 -> 16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 2, 2),     # 16 -> 32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 2, 2),      # 32 -> 64
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(x)


class ResNetAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = encoder
        self.decoder = Decoder()

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z


model = ResNetAE().to(device)
print("Model ready.")

## 8. Combined SSIM + MSE loss (NEW)

In [ ]:
def combined_loss(recon, target, alpha=0.8):
    """
    IMPROVED: alpha * MSE + (1-alpha) * SSIM loss.
    SSIM makes the model sensitive to texture differences where disease appears.
    """
    mse_loss = nn.MSELoss()(recon, target)
    ssim_loss = 1 - ssim(recon, target, data_range=1.0, size_average=True)
    return alpha * mse_loss + (1 - alpha) * ssim_loss

## 9. Augmentation + batch loader (IMPROVED)

In [ ]:
def augment_patch(patch):
    """
    IMPROVED: Random flip + brightness jitter during training.
    Forces AE to learn healthy texture, not specific orientations/lighting.
    """
    if random.random() > 0.5:
        patch = patch[:, ::-1, :].copy()      # horizontal flip
    if random.random() > 0.5:
        patch = patch[::-1, :, :].copy()      # vertical flip
    factor = random.uniform(0.85, 1.15)       # brightness jitter
    return np.clip(patch * factor, 0, 1).astype("float32")


def load_batch(paths, batch_size=16, augment=True):
    batch_paths = random.sample(paths, min(batch_size, len(paths)))
    imgs = [preprocess(p) for p in batch_paths]
    imgs = [i for i in imgs if i is not None]

    patches = []
    for img in imgs:
        p, _ = extract_patches(img, size=64, stride=32)
        if augment:
            p = np.array([augment_patch(patch) for patch in p])
        patches.append(p)

    if len(patches) == 0:
        return None

    patches = np.concatenate(patches, axis=0)
    patches = torch.tensor(patches).permute(0, 3, 1, 2).float()
    return patches

## 10. Training with LR scheduler (IMPROVED)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


def train(model, healthy_paths, epochs=30, steps_per_epoch=200):
    """
    IMPROVED:
    - Combined SSIM+MSE loss
    - Augmentation during batch loading
    - StepLR scheduler halves LR every 10 epochs
    - 30 epochs default (was 20)
    """
    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for step in range(steps_per_epoch):
            patches = load_batch(healthy_paths, augment=True)
            if patches is None:
                continue
            patches = patches.to(device)

            # Denoising: add small noise to input
            noise = torch.randn_like(patches) * 0.05
            noisy = torch.clamp(patches + noise, 0.0, 1.0)

            recon, z = model(noisy)

            # Debug shapes on first step only
            if epoch == 0 and step == 0:
                print(f"Patch shape: {patches.shape} | Recon shape: {recon.shape}")

            loss = combined_loss(recon, patches)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / steps_per_epoch
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.6f} | LR: {current_lr:.2e}")

## 11. Anomaly map — multi-scale (IMPROVED)

In [ ]:
def anomaly_map(model, img):
    """
    IMPROVED: Multi-scale patch extraction.
    Combines errors from 32, 64 and 128px patches for richer anomaly signal.
    """
    model.eval()

    patches, coords, sizes = extract_patches_multiscale(img)

    if len(patches) == 0:
        return np.array([]), []

    patches_t = torch.tensor(patches).permute(0, 3, 1, 2).to(device)

    batch_size = 64
    errors = []
    with torch.no_grad():
        for i in range(0, len(patches_t), batch_size):
            batch = patches_t[i:i+batch_size]
            recon, z = model(batch)
            z_recon = model.encoder(recon)
            err = torch.mean((z - z_recon) ** 2, dim=(1, 2, 3))
            errors.append(err.cpu().numpy())

    errors = np.concatenate(errors)
    return errors, coords, sizes


def rebuild_map(errors, coords, sizes, shape):
    """
    IMPROVED:
    - Handles variable patch sizes
    - Applies Gaussian blur for smoother anomaly maps (fewer false positives)
    """
    h, w = shape[:2]
    anomaly = np.zeros((h, w), dtype=np.float32)
    count   = np.zeros((h, w), dtype=np.float32)

    for err, (x, y), size in zip(errors, coords, sizes):
        y2 = min(y + size, h)
        x2 = min(x + size, w)
        anomaly[y:y2, x:x2] += err
        count[y:y2, x:x2]   += 1

    raw = anomaly / (count + 1e-8)

    # IMPROVED: Gaussian blur merges nearby hotspots → cleaner detections
    smoothed = cv2.GaussianBlur(raw, (21, 21), 0)
    return smoothed

## 12. Adaptive threshold + refined masking (IMPROVED)

In [ ]:
def threshold_map(anomaly, base_pct=85):
    """
    IMPROVED: Adaptive per-image threshold.
    Tightens automatically when an image has high anomaly variance,
    preventing over-segmentation on noisy inputs.
    """
    spread = float(anomaly.max() - anomaly.min())
    # Tighten percentile for high-variance images (more anomalous)
    adjusted_pct = base_pct + min(10.0, spread * 5.0)
    adjusted_pct = min(adjusted_pct, 97.0)
    thresh = np.percentile(anomaly, adjusted_pct)
    return (anomaly > thresh).astype("uint8")


def refine_kmeans(img, mask):
    pixels = img[mask == 1]
    if len(pixels) < 20:
        return mask
    kmeans = KMeans(n_clusters=2, n_init=10).fit(pixels)
    centers = kmeans.cluster_centers_
    # Disease pixels are typically darker/more saturated — pick lower mean
    disease_cluster = np.argmin(np.mean(centers, axis=1))
    refined = np.zeros_like(mask)
    refined[mask == 1] = (kmeans.labels_ == disease_cluster)
    return refined


def clean_mask(mask, min_area=300):
    """
    IMPROVED:
    - Larger kernel (5x5) for stronger noise removal
    - Added MORPH_CLOSE to fill holes inside lesion blobs
    - Raised default min_area to 300px² (was 50)
    """
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)   # remove speckle
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)   # fill inner holes
    mask = cv2.dilate(mask, kernel, iterations=1)
    return mask


def get_boxes(mask, min_area=300):
    mask = mask.astype("uint8")
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h < min_area:
            continue
        boxes.append((x, y, w, h))
    return boxes

## 13. YOLO label helpers

In [ ]:
def create_class_map(root):
    class_map = {}
    idx = 0
    for folder in sorted(os.listdir(root)):
        if "healthy" not in folder.lower() and os.path.isdir(os.path.join(root, folder)):
            class_map[folder] = idx
            idx += 1
    print("Class map:", class_map)
    return class_map


def to_yolo(boxes, shape, class_id):
    h, w = shape[:2]
    labels = []
    for (x, y, bw, bh) in boxes:
        xc = (x + bw / 2) / w
        yc = (y + bh / 2) / h
        bw /= w
        bh /= h
        labels.append([class_id, round(xc, 6), round(yc, 6), round(bw, 6), round(bh, 6)])
    return labels

## 14. Full image processing pipeline

In [ ]:
def process_image(model, img):
    errors, coords, sizes = anomaly_map(model, img)
    if len(errors) == 0:
        return [], np.zeros(img.shape[:2], dtype="uint8"), np.zeros(img.shape[:2])

    anomaly = rebuild_map(errors, coords, sizes, img.shape)
    mask    = threshold_map(anomaly)
    mask    = refine_kmeans(img, mask)
    mask    = clean_mask(mask)
    boxes   = get_boxes(mask)

    return boxes, mask, anomaly


def save_visualization_side_by_side(img, mask, anomaly, save_path):
    """
    IMPROVED: saves original | overlay | anomaly heat-map side by side.
    """
    original = (img * 255).astype("uint8").copy()

    # Red overlay on detected region
    colored = original.copy()
    colored[mask == 1] = [0, 0, 255]
    blended = cv2.addWeighted(original, 0.7, colored, 0.3, 0)

    # Anomaly heat map
    norm = cv2.normalize(anomaly, None, 0, 255, cv2.NORM_MINMAX).astype("uint8")
    heat = cv2.applyColorMap(norm, cv2.COLORMAP_JET)

    combined = np.hstack((original, blended, heat))
    cv2.imwrite(save_path, combined)

## 15. Generate results with visualisations + YOLO labels

In [ ]:
def generate_with_visualization(model, root, viz_dir="visualizations"):
    os.makedirs(viz_dir, exist_ok=True)
    class_map  = create_class_map(root)
    valid_ext  = (".jpg", ".jpeg", ".png", ".bmp")

    total_imgs   = 0
    total_detect = 0

    for folder in os.listdir(root):
        if "healthy" in folder.lower():
            continue

        folder_path = os.path.join(root, folder)
        if not os.path.isdir(folder_path):
            continue

        class_id    = class_map[folder]
        save_folder = os.path.join(viz_dir, folder)
        os.makedirs(save_folder, exist_ok=True)

        for img_name in tqdm(os.listdir(folder_path), desc=f"Processing {folder}"):
            if not img_name.lower().endswith(valid_ext):
                continue

            path = os.path.join(folder_path, img_name)
            img  = preprocess(path)

            if img is None:
                print(f"❌ Failed to load: {path}")
                continue

            total_imgs += 1

            try:
                boxes, mask, anomaly = process_image(model, img)
            except Exception as e:
                print(f"❌ Processing error {path}: {e}")
                continue

            # Save visualisation
            save_path = os.path.join(save_folder, img_name)
            try:
                save_visualization_side_by_side(img, mask, anomaly, save_path)
            except Exception as e:
                print(f"❌ Viz failed {path}: {e}")
                continue

            if not boxes:
                continue

            total_detect += 1

            # Save YOLO label
            try:
                labels   = to_yolo(boxes, img.shape, class_id)
                base, _  = os.path.splitext(path)
                txt_path = base + ".txt"
                with open(txt_path, "w") as f:
                    for lbl in labels:
                        f.write(" ".join(map(str, lbl)) + "\n")
            except Exception as e:
                print(f"❌ Label save failed {path}: {e}")

    print(f"\n✅ Done: {total_detect}/{total_imgs} images had detections.")

## 16. Zip results for download (NEW)

In [ ]:
def zip_results(viz_dir="visualizations", zip_name="results.zip"):
    """
    Zips the entire visualizations folder so you can download it
    from Google Colab in one click.
    """
    print(f"Zipping '{viz_dir}' → '{zip_name}' ...")
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for dirpath, dirnames, filenames in os.walk(viz_dir):
            for filename in filenames:
                filepath = os.path.join(dirpath, filename)
                arcname  = os.path.relpath(filepath, start=os.path.dirname(viz_dir))
                zf.write(filepath, arcname)
    size_mb = os.path.getsize(zip_name) / 1e6
    print(f"✅ Zip created: {zip_name}  ({size_mb:.1f} MB)")
    return zip_name


def download_results(zip_name="results.zip"):
    """Trigger browser download from Google Colab."""
    try:
        from google.colab import files
        files.download(zip_name)
        print("Download started!")
    except ImportError:
        print(f"Not in Colab — find your file at: {os.path.abspath(zip_name)}")

## 17. ▶️ Run everything

In [ ]:
root = "data_set"

# 1. Split dataset
healthy_paths, _ = split_dataset(root)

# 2. Train autoencoder
train(model, healthy_paths, epochs=30, steps_per_epoch=200)

# 3. Generate detections + visualisations
generate_with_visualization(model, root, viz_dir="visualizations")

# 4. Zip results
zip_results(viz_dir="visualizations", zip_name="results.zip")

# 5. Download zip directly in Colab
download_results("results.zip")

## 18. (Optional) Save & reload model checkpoint

In [ ]:
# Save
torch.save(model.state_dict(), "model.pth")
print("Model saved to model.pth")

# Reload later with:
# model = ResNetAE().to(device)
# model.load_state_dict(torch.load("model.pth", map_location=device))
# model.eval()